<img src="https://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

# Python for Finance (2nd ed.)

**Mastering Data-Driven Finance**

&copy; Dr. Yves J. Hilpisch | The Python Quants GmbH

<img src="https://hilpisch.com/images/py4fi_2nd_shadow.png" width="300px" align="left">

# Financial Time Series

In [ ]:
import numpy as np
import pandas as pd
from pylab import mpl, plt
plt.style.use('seaborn-v0_8')
mpl.rcParams['font.family'] = 'serif'
%config InlineBackend.figure_format = 'svg'

In [ ]:
import warnings
warnings.simplefilter('ignore')

## Financial Data

### Data Import

In [ ]:
filename = '../../source/tr_eikon_eod_data.csv'  

In [ ]:
f = open(filename, 'r')  
f.readlines()[:5]  

In [ ]:
data = pd.read_csv(filename,  
                   index_col=0, 
                   parse_dates=True)  

In [ ]:
data.info()  

In [ ]:
data.head()  

In [ ]:
data.tail()  

In [ ]:
data.plot(figsize=(10, 12), subplots=True);  

In [ ]:
instruments = ['Apple Stock', 'Microsoft Stock',
               'Intel Stock', 'Amazon Stock', 'Goldman Sachs Stock',
               'SPDR S&P 500 ETF Trust', 'S&P 500 Index',
               'VIX Volatility Index', 'EUR/USD Exchange Rate',
               'Gold Price', 'VanEck Vectors Gold Miners ETF',
               'SPDR Gold Trust']

In [ ]:
for ric, name in zip(data.columns, instruments):
    print('{:8s} | {}'.format(ric, name))

### Summary Statistics

In [ ]:
data.info()  

In [ ]:
data.describe().round(2)  

In [ ]:
data.mean()  

In [ ]:
data.aggregate([min,  
                np.mean,  
                np.std,  
                np.median,  
                max]  
).round(2)

### Changes Over Time

In [ ]:
data.diff().head()  

In [ ]:
data.diff().mean()  

## Percentage changes
From a statistics point of view, absolute changes are not optimal because they
are dependent on the scale of the time series data itself. Therefore, percentage
changes are usually preferred. The following code derives the percentage changes
or percentage returns (also: simple returns) in a financial context and
visualizes their mean values per column.

In [ ]:
data.pct_change().round(3).head()  

In [ ]:
data.pct_change().mean().plot(kind='bar', figsize=(10, 6));  

### Logarithmic change
As an alternative to percentage returns, log returns can be used. In some
scenarios, they are easier to handle and therefore often preferred in a financial
context. Figure shows the cumulative log returns for the single financial
time series. This type of plot leads to some form of normalization:

In [ ]:
rets = np.log(data / data.shift(1))  

In [ ]:
rets.head().round(3)  

In [ ]:
rets.cumsum().apply(np.exp).plot(figsize=(10, 6));  

### Resampling
Resampling is an important operation on financial time series data. Usually this
takes the form of downsampling, meaning that, for example, a tick data series is
resampled to one-minute intervals or a time series with daily observations is
resampled to one with weekly or monthly observations.

### AVOIDING FORESIGHT BIAS
When resampling, pandas takes by default in many cases the left label (or index value) of the
interval. To be financially consistent, make sure to use the right label (index value) and in
general the last available data point in the interval. Otherwise, a foresight bias might sneak into
the financial analysis.3

In [ ]:
data.resample('1w', label='right').last().head()  

In [ ]:
data.resample('1m', label='right').last().head()  

This plots the cumulative log returns over time: first, the cumsum() method
is called, then np.exp() is applied to the results; finally, the resampling
takes place.

In [ ]:
rets.cumsum().apply(np.exp). resample('1m', label='right').last(
                          ).plot(figsize=(10, 6));

## Rolling Statistics
It is financial tradition to work with rolling statistics, often also called financial
indicators or financial studies. Such rolling statistics are basic tools for financial
chartists and technical traders, for example. This section works with a single
financial time series only.

In [ ]:
sym = 'AAPL.O'
data = pd.DataFrame(data[sym]).dropna()

In [ ]:
data.tail()

### An Overview
It is straightforward to derive standard rolling statistics with pandas.

**EWMA** = Exponentially Weighted Moving Average

In [ ]:
window = 20  

In [ ]:
data['min'] = data[sym].rolling(window=window).min()  

In [ ]:
data['mean'] = data[sym].rolling(window=window).mean()  

In [ ]:
data['std'] = data[sym].rolling(window=window).std()  

In [ ]:
data['median'] = data[sym].rolling(window=window).median()  

In [ ]:
data['max'] = data[sym].rolling(window=window).max()  

In [ ]:
data['ewma'] = data[sym].ewm(halflife=0.5, min_periods=window).mean()  

To derive more specialized financial indicators, additional packages are
generally needed (see, for instance, the financial plots with Cufflinks in
“Interactive 2D Plotting”). Custom ones can also easily be applied via the
apply() method.

The following code shows a subset of the results and visualizes a selection of the calculated rolling statistics:

In [ ]:
data.dropna().head()

In [ ]:
window = 300

ax = data[['min', 'mean', 'max']].iloc[-window:].plot(
    figsize=(10, 6), style=['g--', 'r--', 'g--'], lw=0.8)

data[sym].iloc[-window:].plot(ax=ax, lw=2.0);  

### A Technical Analysis Example
Rolling statistics are a major tool in the so-called technical analysis of stocks, as
compared to the fundamental analysis which focuses, for instance, on financial
reports and the strategic positions of the company whose stock is being
analyzed.

**A decades-old trading strategy based on technical analysis is using two simple
moving averages (SMAs).**
</br>The idea is that the trader should go
* long on a stock (or financial instrument in general) when the shorter-term SMA is above the
longer-term SMA
* short when the opposite holds true

The concepts can be made precise with pandas and the capabilities of the DataFrame
object.

Rolling statistics are generally only calculated when there is enough data given
the window parameter specification.

As Figure shows, the SMA time series
only start at the day for which there is enough data given the specific
parameterization:

In [ ]:
data['SMA1'] = data[sym].rolling(window=42).mean()

In [ ]:
data['SMA2'] = data[sym].rolling(window=252).mean()

In [ ]:
data[[sym, 'SMA1', 'SMA2']].tail()

In [ ]:
data[[sym, 'SMA1', 'SMA2']].plot(figsize=(10, 6));

In [ ]:
data.dropna(inplace=True)  

In [ ]:
data['positions'] = np.where(data['SMA1'] > data['SMA2'],
                             +1,
                             -1)

In [ ]:
ax = data[[sym, 'SMA1', 'SMA2', 'positions']].plot(figsize=(10, 6),
                                              secondary_y='positions')

ax.get_legend().set_bbox_to_anchor((0.25, 0.85));

The trading strategy implicitly derived here only leads to a few trades per se:
only when the position value changes (i.e., a crossover happens) does a trade
take place. Including opening and closing trades, this would add up to just six
trades in total.

## Regression Analysis
### Correlation
As a further illustration of how to work with pandas and financial time series
data, consider the case of the S&P 500 stock index and the VIX volatility index.

**It is a stylized fact that when the S&P 500 rises, the VIX falls in general, and
vice versa. This is about correlation and not causation.**

This section shows how
to come up with some supporting statistical evidence for the stylized fact that the
S&P 500 and the VIX are (highly) negatively correlated.

### The Data

In [ ]:
# EOD data from Thomson Reuters Eikon Data API
raw = pd.read_csv('../../source/tr_eikon_eod_data.csv',
                 index_col=0, parse_dates=True)

In [ ]:
data_regr = raw[['.SPX', '.VIX']].dropna()

In [ ]:
data_regr.tail()

In [ ]:
data_regr.plot(subplots=True, figsize=(10, 6));

When plotting (parts of) the two time series in a single plot and with adjusted
scalings, the stylized fact of negative correlation between the two indices
becomes evident through simple visual inspection 

In [ ]:
data_regr.loc[:'2012-12-31'].plot(secondary_y='.VIX', figsize=(10, 6));

### Log Returns
As pointed out earlier, **statistical analysis in general relies on returns instead of
absolute changes or even absolute values**.

Therefore, we’ll calculate log returns
first before any further analysis takes place.

Figure shows the high
variability of the log returns over time. For both indices so-called “volatility
clusters” can be spotted.

**In general, periods of high volatility in the stock index
are accompanied by the same phenomena in the volatility index:**

In [ ]:
rets = np.log(data_regr / data_regr.shift(1))

In [ ]:
rets.head()

In [ ]:
rets.dropna(inplace=True)

In [ ]:
rets.plot(subplots=True, figsize=(10, 6));

In such a context, the `pandas.scatter_matrix()` plotting function comes in
handy for visualizations.

It plots the log returns of the two series against each
other, and one can add either a histogram or a kernel density estimator (KDE) on
the diagonal:

In [ ]:
sm = pd.plotting.scatter_matrix(rets,
                                alpha=0.2,
                                diagonal='hist',
                                hist_kwds={'bins': 35},
                                figsize=(10, 6));

sci_not = False  # baah
import matplotlib.ticker as mticker
for ax in sm.flatten():
    continue
    if sci_not:
        # Force scientific notation for all subplots
        formatter = mticker.ScalarFormatter()
    else:
        N = 1  # Set N to your desired number of decimal places
        # Apply fixed precision formatting to all subplots
        formatter = mticker.FormatStrFormatter(f'%.{N}f')
    # Explicitly set the formatter to ScalarFormatter first
    ax.xaxis.set_major_formatter(formatter)
    ax.yaxis.set_major_formatter(formatter)

    if sci_not:  # Force scientific notation for all subplots
        # Now apply the scientific notation style
        ax.ticklabel_format(style='sci', scilimits=(0, 0), axis='both')

### OLS Regression
With all these preparations, an ordinary least-squares (OLS) regression analysis
is convenient to implement.

Figure shows a scatter plot of the log returns
and the linear regression line through the cloud of dots.

**The slope is obviously
negative, providing support for the stylized fact about the negative correlation
between the two indices**:

In [ ]:
reg = np.polyfit(rets['.SPX'], rets['.VIX'], deg=1)

print(f'fitting y = {reg[0]:0.3f} * x + {reg[1]:0.3f}')

In [ ]:
ax = rets.plot(kind='scatter', x='.SPX', y='.VIX', figsize=(10, 6))

ax.plot(rets['.SPX'], np.polyval(reg, rets['.SPX']), 'r', lw=2);  

### Correlation
Finally, we consider correlation measures directly. Two such measures are
considered:
* a static one taking into account the complete data set
* a rolling one showing the correlation for a fixed window over time.

Figure illustrates that **the correlation indeed varies over time but that it is always, given the
parameterization, negative**.

This provides strong support for the stylized fact that
the S&P 500 and the VIX indices are (strongly) negatively correlated:

In [ ]:
# The correlation matrix for the whole DataFrame
rets.corr()

In [ ]:
corr = rets['.SPX'].rolling(window=252).corr(rets['.VIX'])

ax = corr.plot(figsize=(10, 6))

# adds the static value to the plot as horizontal line
ax.axhline(rets.corr().iloc[0, 1], c='r');

## High Frequency Data
Tick data sets are a special case of financial time series.</br>
Frankly, **they can be handled more or
less in the same ways as, for instance, the EOD data set used throughout this
chapter so far**.

Importing such data sets also is quite fast in general with pandas.
The data set used comprises 17,352 data rows:

In [ ]:
# from fxcmpy import fxcmpy_tick_data_reader as tdr
# data = tdr('EURUSD', start='2018-6-25', end='2018-06-30')
# data.get_data(start='2018-6-29',
#               end='2018-06-30').to_csv('../../source/fxcm_eur_usd_tick_data.csv')

In [ ]:
%%time
# data from FXCM Forex Capital Markets Ltd.
tick = pd.read_csv('../../source/fxcm_eur_usd_tick_data.csv',
                     index_col=0, parse_dates=True)

In [ ]:
tick.info()

In [ ]:
tick['Mid'] = tick.mean(axis=1)  

In [ ]:
tick['Mid'].plot(figsize=(10, 6));

Working with tick data is generally a scenario where resampling of financial
time series data is needed.

The code that follows resamples the tick data to five-minute bar data, which can then be used, for example, to
backtest algorithmic trading strategies or to implement a technical analysis:

In [ ]:
tick_resam = tick.resample(rule='5min', label='right').last()

In [ ]:
tick_resam.head()

In [ ]:
tick_resam['Mid'].plot(figsize=(10, 6));

<img src="https://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

<a href="https://tpq.io" target="_blank">https://tpq.io</a> | <a href="https://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:training@tpq.io">training@tpq.io</a>